# Oxygen dissociation at fixed enthalpy and pressure

Starting with heated molecular oxygen, find the equilibrium temperature and composition for the neutral O₂/O gas. Dissociation draws energy from the inlet sensible enthalpy. The input file uses a small CEA-style HP deck subset.

In [ ]:
from pathlib import Path
import re
import math
import numpy as np
import cea

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "config/oxygen_hp.inp").is_file())
deck = (root / "config/oxygen_hp.inp").read_text()
number = r"([0-9]+(?:\.[0-9]+)?)"
pattern = (r"problem\s+hp\s+p,bar=" + number +
           r"\s+reactants\s+name=O2\s+moles=" + number +
           r"\s+t,k=" + number + r"\s+end")
fields = re.fullmatch(pattern, deck.strip(), re.I)
if fields is None:
    raise ValueError("Expected an HP deck with one O2 inlet.")
pressure, inlet_moles, inlet_temperature = map(float, fields.groups())
if min(pressure, inlet_moles, inlet_temperature) <= 0:
    raise ValueError("Pressure, moles and temperature must be positive.")

In [ ]:
feed = cea.Mixture(["O2"])
products = cea.Mixture(["O2", "O"])
weights = feed.weights_from_moles(np.array([inlet_moles]))
enthalpy = feed.calc_property(cea.ENTHALPY, weights, inlet_temperature)
solver = cea.EqSolver(products, reactants=feed)
state = cea.EqSolution(solver)
solver.solve(state, cea.HP, enthalpy / cea.R, pressure, weights)
assert state.converged
fractions = state.mole_fractions
assert math.isclose(sum(fractions.values()), 1, abs_tol=1e-6)
assert 0 < state.T <= inlet_temperature + 1e-4
assert math.isclose(state.P, pressure, rel_tol=1e-6)
print(f"Equilibrium temperature: {state.T:.2f} K at {state.P:g} bar")
for species, fraction in fractions.items():
    print(f"{species:4s} {fraction:.8f}")

The pressure is in bar. The enthalpy argument uses the normalization in the [NASA Python sample](https://github.com/nasa/cea/blob/main/source/bind/python/cea/samples/h2_o2.py). See the [CEA Python API](https://nasa.github.io/cea/interfaces/python_api.html) for solution properties.